In [1]:
# EXTENSION (USING FUNCTIONS FROM PART 1 AND 2)

# Changes made to part 1 and 2 functions are as follows:

# frequency_analysis now uses the precomputed English frequencies, so they aren't rebuilt for every call
# Changed the way frequency analysis is peformed, just looking at the most frequent letter in a piece of cipher text
# Changed decrypt vigenere so that we can try to decrypt without a key, but using a key to test the decryption function itself is still permitted, as well as return information on what happened during the process
# Changed test vigenere to report what type of failure occurs as well as success

# New functions added are: find_vigenere_key_length, split_text_by_key_length and run_trials

import random
import math

ascii_lowercase = "abcdefghijklmnopqrstuvwxyz"
ascii_uppercase = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"

def get_plaintext(txt_file_name):
    # open the file
    with open(txt_file_name, "r") as file:
        text = file.read()
    plaintext = ""
    # iterate through and only add letters to the plaintext
    for i in text:
        if i in ascii_lowercase or i in ascii_uppercase:
            plaintext += i
    return plaintext

# Import books.
book1 = get_plaintext("moby_dick.txt")
book2 = get_plaintext("tale_of_two_cities.txt")
book3 = get_plaintext("treasure_island.txt")
book4 = get_plaintext("wuthering_heights.txt")
book5 = get_plaintext("pride_and_prejudice.txt")
book6 = get_plaintext("pygmailion.txt")    
# Concatenate into one "master text" we can analyse.
master_text = book1 + book2 + book3 + book4 + book5 + book6 
   
def get_english_language_frequencies(master_text):   
    # Calculate the frequency of each letter (uppercase or lowercase) in a master text.
    letter_frequencies = []
    for i in range(26):
        letter_sum = master_text.count(ascii_lowercase[i]) + master_text.count(ascii_uppercase[i])
    # Add to a list of tuples, the first element corresponding to the position of a letter in the alphabet and the second to the frequency.
        letter_frequencies.append((i, letter_sum / len(master_text)))
    
    # Now sort.
    return sorted(letter_frequencies, key=lambda x: x[1]) 
    # key=lambda x: x[1] allows us to sort the tuples by the second element of the tuple; i.e. the frequency.

#We get our list of letter frequencies for the English language and declare it globally
letter_frequencies = get_english_language_frequencies(master_text)
    
def frequency_analysis(ciphertext):
    # Count the number of times each letter a-z appears in the ciphertext.
    ciphertext_letter_frequencies = []
    for i in range(len(ascii_lowercase)):
        ciphertext_letter_sum = ciphertext.count(ascii_lowercase[i]) + ciphertext.count(ascii_uppercase[i])
        ciphertext_letter_frequencies.append((i,ciphertext_letter_sum / len(ciphertext)))
    # Now sort
    ciphertext_letter_frequencies = sorted(ciphertext_letter_frequencies, key=lambda x: x[1]) 

    # The key is given by the alphabetic shift between letters of the same frequency.    
    # We just use the shift determined by what lines up with e, the most common letter in the English  language
    key = ((ciphertext_letter_frequencies[25][0] - letter_frequencies[25][0])%26)
    return key

def caesar_cipher(plaintext, key):
    cipher_text = ""
    # Iterate through each letter in the plaintext
    for i in plaintext:
        # Shift by the value of the key.
        if i in ascii_lowercase:
            index = ascii_lowercase.index(i)
            cipher_text += ascii_lowercase[(index + key) % 26]
        else:
            index = ascii_uppercase.index(i)
            cipher_text += ascii_uppercase[(index + key) % 26]
    return cipher_text

# And decrypt...
def decrypt_caesar_cipher(ciphertext, key):
    return caesar_cipher(ciphertext, -1*key)
    
def vigenere_cipher(plaintext, key):
    # Note that for the vigenere cipher, the key is a word.
    length = len(key)
    cipher_text = ''
    # Iterate through each letter in the plaintext
    for i in range(len(plaintext)):
        # Work out the corresponding caesar key to the letter
        number = ascii_lowercase.index((key[i % length].lower()))
        # Compute the new letter
        cipher_text += caesar_cipher(plaintext[i], number)
    return cipher_text

def find_vigenere_key_length(ciphertext, verbose = False):    
    """Given a piece of text, this routine finds repeated 3 grams of text,
        and calculates the distance between them. It then uses a voting system 
        to decide on a candidate for the key length, using math.gcd to aid in this."""
    # We declare three empty lists to keep track as we iterate through the text
    unique_3_grams = []
    length_array = []
    repeated_3_grams = []   
    for i in range(len(ciphertext)-2):
        # Look at the current 3-gram
        current_3_gram = ciphertext[i:i+3].lower()
        # If it is new, then we look for repeats in the text
        if current_3_gram not in unique_3_grams:
            unique_3_grams.append(current_3_gram)
            last_instance = i
            # We deal fully with a unique 3-gram once we have found it
            for j in range(i+1, len(ciphertext)-2):               
                text_slice = ciphertext[j:j+3].lower()
                # If we find a repeat in the text, then we add it to our repeat array, and record the distance from the last repeat
                if text_slice == current_3_gram:
                    if text_slice not in repeated_3_grams:
                        repeated_3_grams.append(text_slice)
                        # We add the distance between occurrences to the length array
                        length_array.append(j-last_instance)
                    # However, if there are more than two occurrences of a 3-gram, we take the gcd of the distances between each occurrence
                    # They all fold in to provide one distance per 3-gram, which we will use to vote on a final candidate
                    else:
                        length_array[-1] = math.gcd(length_array[-1], j-last_instance)
                    # Update the last instance to find the next distance
                    last_instance = j  
    # We remove any candidates that are less than length 3, because these will most likely be chance repeats that could affect the calculation. 
    # We also remove proposals for very large key lengths. This is because they will not yield enough characters for successful frequency analysis (reject keys resulting in less than 20 characters).
    length_array = [x for x in length_array if 3 <= x <= len(ciphertext) // 20]
    if verbose:
        print ("The key length candidates are: {}".format(length_array))
    # Each candidate for key length now votes on its divisors, provided we have been able to find repeated 3-grams
    if len(length_array) != 0:     
        # We create a list of 0s, which will be used to keep track of votes for divisors
        votes = [0] * (max(length_array) + 1)
        # For each potential key length, we add a vote for each of its divisors
        for candidate in length_array:
            for i in range(3,candidate + 1):
                if candidate % i == 0:
                    votes[i] += 1
        # The value with the most votes will either be the key or a factor of the key length, set the amount of votes to best
        best = max(votes)
        # We then work backwards through our votes list. To find our candidate for the key length, we use a tolerance of 0.8.
        # The largest divisor that has enough votes within this tolerance will be the value we choose for the key length.
        # This tolerance helps us identify key lengths which have smaller key factors that would beat them in the vote
        for i in range(len(votes) -1, 2, -1):
            # Note: 0.8 was found through testing to have a good success rate, 0.9 was too strict and let smaller divisors beat the true length frequently
            if (votes[i] >= 0.8 * best):
                if verbose:
                    print("The supposed key length is:", i)
                # Will always return something as "best" will satisfy this by definition
                return i
    # If no useable candidates are found then we return -1
    else:
        return -1
       
def split_text_by_key_length(ciphertext,key_length):
    """Given a piece of text and a key length, this routine returns a list of strings, whose characters are of key length apart in the text. 
        These will then be analysed individually with frequency analysis to decrypt the vigenere cipher."""
    # The list of strings we will return
    char_strings = []
    for i in range(key_length):      
        all_found = False
        index = i
        # One string will be constructed for every value in the key. We add the first character to start.
        current_string = ciphertext[i]
        # This loop will assemble all the characters of keylength apart from a given starting point
        while all_found == False:
            if (index+key_length >= len(ciphertext)):
                all_found = True
            else:
                index = index + key_length
                current_string += ciphertext[index]
        char_strings.append(current_string)
    # Check we have a string to analyse for each shift from the key
    assert(len(char_strings) == key_length)
    return(char_strings)

def decrypt_vigenere_cipher(ciphertext, key = None, verbose = False):
    """Given a piece of ciphertext only, this method first will call functions to perform a Kasiski analysis on the ciphertext, calling the functions defined above to find a candidate for the key.
        If a key is provided, this step will be skipped. Then using the key either found or provided, the cipher text will be decrypted and the plaintext returned."""
    # If no key was provided to decrypt, we perform a Kasiski style analysis to find a candidate
    if key is None:
        # We find a key length guess using find_vigenere_key_length
        key_length_guess = find_vigenere_key_length(ciphertext,verbose = verbose)
        # If the method fails and returns -1, we return None
        if key_length_guess == -1:
            if verbose:                    
                print("No useable key length found.")
            return (None, -1)
        # We split our text using our key length guess and get strings we can analyse using frequency analysis
        char_strings = split_text_by_key_length(ciphertext,key_length_guess)
        # This will be added to as we perform frequency analysis on our character strings, giving us our guess for the key
        key_string = ""
        # Loop through and perform frequency analysis on each string
        for char_string in char_strings:
            key = frequency_analysis(char_string)
            key_string += str(ascii_lowercase[key])
        # Set our value of key to this string
        key = key_string 
        if verbose:              
            print("The key found by Kasiski analysis is: {}".format(key_string))
    # Now, we either use the key found using Kasiski analysis, or the one given        
    length = len(key)
    plaintext = ''
    # Iterate through each letter in the ciphertext
    for i in range(len(ciphertext)):
        # Work out the corresponding caesar key to the letter
        number = ascii_lowercase.index((key[i % length].lower()))
        # Decrypt the letter
        plaintext += decrypt_caesar_cipher(ciphertext[i], number)
    # Return the decrypted plaintext
    return plaintext, length


def test_vigenere(plaintext, key_length, use_key = True, verbose = False):
    """This function will test the ability to encrypt and decrypt a piece of text using the vigenere cipher. It will generate a random key of key_length, passed by the user.
        It will then encrypt the plaintext using this key. To decrypt, if use_key is true, it will just use the key we generated to decrypt and check it is equal to the plaintext.
        Otherwise, Kasiski analysis will be performed to achieve a guess for the key, which will be used to decrypt. Again this will be checked against the plaintext.
        Tuples returned to track failure types"""
    key = ''
    # Generating some random key to encrypt with
    for i in range(key_length):
        key += ascii_lowercase[random.randint(0, 25)]
    if verbose:           
        print("The true key is {}.".format(key))
    # Encrypt the message
    ciphertext_test =  vigenere_cipher(plaintext, key)
    # Decrypt with true key if we want to just check function of the decrypt method
    if use_key == True:      
        plaintext_test, guessed_key_length = decrypt_vigenere_cipher(ciphertext_test, key, verbose = verbose)
    # Find and decrypt with the key from Kasiski analysis
    else:
        plaintext_test, guessed_key_length = decrypt_vigenere_cipher(ciphertext_test, key = None, verbose = verbose)
    # Error message if we couldn't perform the analysis, and a value to indicate failure type
    if (plaintext_test) is None or (guessed_key_length != key_length):
        return ('Kasiski analysis failed', 0)
    # Finally, we determine the outcome of the encryption/decryption by comparing it to the original plaintext
    elif plaintext_test == plaintext:
        return ('Test passed', 1)
    return ('Test failed, key = ' + key, 2)


#TESTING

#First we test that the process vigenere encryption and decryption with the same key works on the text
print(test_vigenere(get_plaintext('message_emma.txt'),6, use_key=True, verbose = True)[0])
print()
#We test the same piece of text again, decrypting without using the encryption key and using Kasiski analysis instead
print(test_vigenere(get_plaintext('message_emma.txt'),6, use_key=False, verbose = False)[0])
print()

#We now perform random tests on different random chunks of text from the master text, and observe how different changes affect the reliability of our algorithm
def run_trials(text_length, key_length, num_trials):
    #Initialise counters to track failure types
    passed = kasiski_failures = failed = 0
    for i in range(num_trials): 
        #Get a random chunk of text of the specified length
        upper = random.randint(text_length,len(master_text))
        lower = upper - text_length
        #Finds out if we succeeded or how we failed, and adds one to the corresponding tally
        val = test_vigenere(master_text[lower:upper],key_length, use_key=False, verbose = False)[1]
        if val == 0:
            kasiski_failures += 1
        elif val == 1:
            passed += 1
        else:
            failed += 1
    print("Over {} trials, for a text of length {} and a key of length {}, we had the following results:".format(num_trials, text_length, key_length))
    print("The process was successful {} times.\nKasiski analysis of positions failed {} times.\nFrequency analysis failed {} times.".format(passed,kasiski_failures, failed))
    print()

#We carry out some tests and observe results
#Note that taking text from the master text does flatter the frequency step a bit, but it is the easiest way for us to test large random chunks of english text in this code
run_trials(text_length = 100, key_length = 6, num_trials = 20)
run_trials(text_length = 1000, key_length = 6, num_trials = 20)
run_trials(text_length = 5000, key_length = 6, num_trials = 10)
run_trials(text_length = 5000, key_length = 15, num_trials = 10)
run_trials(text_length = 10000, key_length = 6, num_trials = 10)
        

The true key is xpupjf.
Test passed

Test passed

Over 20 trials, for a text of length 100 and a key of length 6, we had the following results:
The process was successful 0 times.
Kasiski analysis of positions failed 20 times.
Frequency analysis failed 0 times.

Over 20 trials, for a text of length 1000 and a key of length 6, we had the following results:
The process was successful 4 times.
Kasiski analysis of positions failed 0 times.
Frequency analysis failed 16 times.

Over 10 trials, for a text of length 5000 and a key of length 6, we had the following results:
The process was successful 8 times.
Kasiski analysis of positions failed 0 times.
Frequency analysis failed 2 times.

Over 10 trials, for a text of length 5000 and a key of length 15, we had the following results:
The process was successful 1 times.
Kasiski analysis of positions failed 5 times.
Frequency analysis failed 4 times.

Over 10 trials, for a text of length 10000 and a key of length 6, we had the following results:
